# Per-player action classifier (Volleyball dataset)

The capstone action detector does not generalise to broadcast footage of other matches: on the Volleyball dataset
test split it almost never predicts set or receive (docs/results/actions.md). This notebook trains a classifier on
player crops instead: the 9 person actions of the Volleyball dataset (Ibrahim et al., CVPR 2016), official split by
match, crops built by `scripts/build_volleyball_actions.py` (key frame and +-2, +-4 frames for train / val, key frame
only for test; 'standing' thinned to 20% of train / val). Test accuracy is comparable to the person action accuracy
reported in the literature with ground-truth boxes (e.g. 82.4%, Bagautdinov et al., CVPR 2017).

Runs on **Kaggle** (GPU, Internet on) with input `sherif31/group-activity-recognition-volleyball`.
Outputs in `/kaggle/working/`: weights, test metrics and per-crop test probabilities.

In [ ]:
import glob, json, os, shutil, sys
WORK = '/tmp/work'; OUT = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
VIDEOS = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/videos/47', recursive=True))
PKL = glob.glob('/kaggle/input/**/annot_all.pkl', recursive=True)[0]
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
!pip -q install -e {WORK}/repo --no-deps
!pip -q install ultralytics

In [ ]:
DATA = f'{WORK}/vb_actions'
!cd {WORK}/repo && python scripts/build_volleyball_actions.py --videos {VIDEOS} --pkl {PKL} --out {DATA}

In [ ]:
from ultralytics import YOLO
RUN = 'person_action_cls'
for weights in ('yolo26s-cls.pt', 'yolo11s-cls.pt'):   # YOLO26 if this ultralytics has it
    try:
        model = YOLO(weights); break
    except Exception as e:
        print('not available:', weights, e)
model.train(data=DATA, epochs=30, imgsz=224, batch=128, patience=8, project=f'{OUT}/runs', name=RUN, exist_ok=True,
            fliplr=0.5, degrees=0, plots=True)

In [ ]:
import numpy as np
from pathlib import Path
best = f'{OUT}/runs/{RUN}/weights/best.pt'
m = YOLO(best)
names = m.names
rows, y, p = [], [], []
for cls_dir in sorted(Path(DATA, 'test').iterdir()):
    files = sorted(cls_dir.glob('*.jpg'))
    for i in range(0, len(files), 256):
        for f, r in zip(files[i:i + 256], m.predict([str(x) for x in files[i:i + 256]], imgsz=224, verbose=False)):
            probs = r.probs.data.cpu().numpy()
            rows.append([f.stem, cls_dir.name] + [round(float(v), 5) for v in probs])
            y.append(cls_dir.name); p.append(names[int(probs.argmax())])
import csv
with open(f'{OUT}/person_action_test_probs.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['crop', 'label'] + [names[i] for i in range(len(names))]); w.writerows(rows)
y, p = np.array(y), np.array(p)
classes = [names[i] for i in range(len(names))]
per_class = {c: float((p[y == c] == c).mean()) for c in classes if (y == c).any()}
res = {'test_crops': int(len(y)), 'accuracy': float((y == p).mean()), 'mean_class_accuracy': float(np.mean(list(per_class.values()))),
       'per_class_accuracy': per_class,
       'confusion': {t: {q: int(((y == t) & (p == q)).sum()) for q in classes} for t in classes}, 'weights': os.path.basename(best)}
print(json.dumps({k: v for k, v in res.items() if k != 'confusion'}, indent=1))
json.dump(res, open(f'{OUT}/person_action_test.json', 'w'), indent=1)
shutil.copy(best, f'{OUT}/person_action_cls.pt')